In [1]:
import pandas as pd
import re
import numpy as np
import seaborn as sn

Matplotlib is building the font cache; this may take a moment.


In [3]:
df=pd.read_csv("carrefour_products.csv")
df.head()

,id,ean,name,type,food_type,brand,product_origin,supplier,top_category,top_category_id,...,seller,seller_type,shipping,is_marketplace,is_express,is_bulk,is_scalable,is_fbc,preorder,has_promo
0,49834,6161100912830,Tropical Heat Spices Pilau Masala Whol100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,Carrefour,main,SLOTTED,False,False,False,False,False,False,False
1,32190,6161100913899,Tropical Heat Spices Oregano Leaves 20G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,Carrefour,main,SLOTTED,False,False,False,False,False,False,True
2,32063,6161100911482,Tropical Heat Spices Chilli Flakes 75G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,Carrefour,main,SLOTTED,False,False,False,False,False,False,True
3,32048,6161100911307,Tropical Heat Spices Cardamoms Ground 100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,Carrefour,main,SLOTTED,False,False,False,False,False,False,False
4,32114,6161100913080,Tropical Heat Turmeric Ground 45G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,Carrefour,main,SLOTTED,False,False,False,False,False,False,False


In [6]:
quick_df= pd.read_csv("C:\\Users\\RodahNambuyaChepkori\\OneDrive - THH LLC\Documents\\quickmart_products.csv")
quick_df.head()

,product_id,name,price_kes,old_price_kes,discount,in_stock,product_url,image_url,category
0,1595554,Blue Band Porridge 250G,40.0,NaN,50.00,True,https://www.quickmart.co.ke/blue-band-porridge...,https://cfn.quickmart.co.ke/resized/230_230/pr...,flour
1,1584617,Blue Band Instant Porridge 500G,80.0,NaN,50.00,True,https://www.quickmart.co.ke/blue-band-instant-...,https://cfn.quickmart.co.ke/resized/230_230/pr...,flour
2,265863,Soko Homebaking Flour 2Kg,149.0,NaN,12.87,True,https://www.quickmart.co.ke/soko-homebaking-fl...,https://cfn.quickmart.co.ke/resized/230_230/pr...,flour
3,319381,Soko Nutrigo 2Kg,271.0,NaN,9.67,True,https://www.quickmart.co.ke/soko-nutrigo-2kg-37,https://cfn.quickmart.co.ke/resized/230_230/pr...,flour
4,267078,Ajab Premium Fortified Sifted Maize Meal 2Kg,162.0,NaN,8.47,True,https://www.quickmart.co.ke/ajab-premium-forti...,https://cfn.quickmart.co.ke/resized/230_230/pr...,flour


In [13]:
# extracting the sku

def extract_sku(product_name):
    match = re.search(r'(\d+\s*(?:KG|G|ml|l|L))\s*$', str(product_name), re.IGNORECASE)
    return match.group(1).replace(" ", "").upper() if match else None

# Apply to dataframe
df['sku'] = df['name'].apply(extract_sku)
# print head
df.head()


,id,ean,name,type,food_type,brand,product_origin,supplier,top_category,top_category_id,...,seller_type,shipping,is_marketplace,is_express,is_bulk,is_scalable,is_fbc,preorder,has_promo,sku
0,49834,6161100912830,Tropical Heat Spices Pilau Masala Whol100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,main,SLOTTED,False,False,False,False,False,False,False,100G
1,32190,6161100913899,Tropical Heat Spices Oregano Leaves 20G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,main,SLOTTED,False,False,False,False,False,False,True,20G
2,32063,6161100911482,Tropical Heat Spices Chilli Flakes 75G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,main,SLOTTED,False,False,False,False,False,False,True,75G
3,32048,6161100911307,Tropical Heat Spices Cardamoms Ground 100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,main,SLOTTED,False,False,False,False,False,False,False,100G
4,32114,6161100913080,Tropical Heat Turmeric Ground 45G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,main,SLOTTED,False,False,False,False,False,False,False,45G


In [14]:
df[['name', 'sku']]

,name,sku
0,Tropical Heat Spices Pilau Masala Whol100G,100G
1,Tropical Heat Spices Oregano Leaves 20G,20G
2,Tropical Heat Spices Chilli Flakes 75G,75G
3,Tropical Heat Spices Cardamoms Ground 100G,100G
4,Tropical Heat Turmeric Ground 45G,45G
...,...,...
3995,Milya's Tutti Frutti Flavoured Popcorn 53g,53G
3996,Taystee Original Red Sauce 400G,400G
3997,Sunveat Oat & Honey Digestive Biscuit 800g,800G
3998,Cadbury Single Creme Egg 40G,40G


In [15]:
import pandas as pd
import re

def normalize_unit(unit):
    unit = unit.upper()
    if unit in ['G', 'GM', 'GMS']:
        return 'G'
    if unit == 'KG':
        return 'KG'
    if unit in ['ML']:
        return 'ML'
    if unit == 'L':
        return 'L'
    return None


def extract_standard_sku(product_name):
    text = str(product_name).upper()

    # 1️⃣ Handle multipack weights (e.g. 10G x 6, 500G X2, 2.5G x Pack of 50)
    multipack = re.search(
        r'(\d+(?:\.\d+)?)\s*(G|GM|GMS|KG|ML|L|gr|Litres)\s*[X×]\s*(?:PACK OF\s*)?(\d+)',
        text
    )

    if multipack:
        qty = float(multipack.group(1))
        unit = normalize_unit(multipack.group(2))
        count = int(multipack.group(3))

        total = qty * count

        # Convert to base units
        if unit == 'KG':
            total *= 1000
            unit = 'G'
        if unit == 'L':
            total *= 1000
            unit = 'ML'

        # Remove decimals where possible
        total = int(total) if total.is_integer() else round(total, 2)

        return f"{total}{unit}"

    # 2️⃣ Handle single pack weight/volume at end
    single = re.search(
        r'(\d+(?:\.\d+)?)\s*(G|GM|GMS|KG|ML|L)\b(?!\s*(PIECE|PCS|COUNT))',
        text
    )

    if single:
        qty = float(single.group(1))
        unit = normalize_unit(single.group(2))

        if unit == 'KG':
            qty *= 1000
            unit = 'G'
        if unit == 'L':
            qty *= 1000
            unit = 'ML'

        qty = int(qty) if qty.is_integer() else round(qty, 2)
        return f"{qty}{unit}"

    # 3️⃣ Ignore counts like "39 Pieces", "100 Count"
    return None


# ✅ APPLY TO DATAFRAME
df['standard_sku'] = df['name'].apply(extract_standard_sku)

In [20]:
import re
import pandas as pd

UNIT_MAP = {
    "G": "G", "GM": "G", "GMS": "G", "GRAM": "G", "GRAMS": "G", "GR": "G",
    "KG": "KG",
    "ML": "ML",
    "L": "L", "LITRE": "L", "LITRES": "L", "LITER": "L", "LITERS": "L"
}

def normalize_unit(unit):
    return UNIT_MAP.get(unit.upper())

def extract_standard_sku(product_name):
    text = str(product_name).upper()

    # ===============================
    # 1️⃣ MULTIPACK (e.g. 70 GR X 5)
    # ===============================
    multipack = re.search(
        r'(\d+(?:\.\d+)?)\s*'
        r'(G|GM|GMS|GRAMS?|GR|KG|ML|L|LITRES?|LITERS?)\s*'
        r'[X×]\s*(\d+)',
        text
    )

    if multipack:
        qty = float(multipack.group(1))
        unit = normalize_unit(multipack.group(2))
        count = int(multipack.group(3))

        total = qty * count

        if unit == "KG":
            total *= 1000
            unit = "G"
        elif unit == "L":
            total *= 1000
            unit = "ML"

        total = int(total) if total.is_integer() else round(total, 2)
        return f"{total}{unit}"

    # =====================================
    # 2️⃣ SINGLE WEIGHT / VOLUME ANYWHERE
    # =====================================
    single = re.search(
        r'(\d+(?:\.\d+)?)\s*'
        r'(G|GM|GMS|GRAMS?|GR|KG|ML|L|LITRES?|LITERS?)\b',
        text
    )

    if single:
        qty = float(single.group(1))
        unit = normalize_unit(single.group(2))

        if unit == "KG":
            qty *= 1000
            unit = "G"
        elif unit == "L":
            qty *= 1000
            unit = "ML"

        qty = int(qty) if qty.is_integer() else round(qty, 2)
        return f"{qty}{unit}"

    # ===============================
    # 3️⃣ IGNORE COUNTS / PIECES
    # ===============================
    return None


# ✅ APPLY TO DATAFRAME
df['standard_sku_2'] = df['name'].apply(extract_standard_sku)

In [21]:
#saving to confirm
df.to_csv("caf_sku.csv")

In [22]:
import pandas as pd
import numpy as np
import re

def split_sku(sku):
    if sku is None:
        return pd.Series([None, None])

    match = re.match(r'(\d+(?:\.\d+)?)(G|ML)', sku)
    if match:
        return pd.Series([float(match.group(1)), match.group(2)])

    return pd.Series([None, None])


df[['sku_qty', 'sku_unit']] = df['standard_sku_2'].apply(split_sku)

In [23]:
df.head()

,id,ean,name,type,food_type,brand,product_origin,supplier,top_category,top_category_id,...,is_bulk,is_scalable,is_fbc,preorder,has_promo,sku,standard_sku,standard_sku_2,sku_qty,sku_unit
0,49834,6161100912830,Tropical Heat Spices Pilau Masala Whol100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,False,False,100G,100G,100G,100.0,G
1,32190,6161100913899,Tropical Heat Spices Oregano Leaves 20G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,False,True,20G,20G,20G,20.0,G
2,32063,6161100911482,Tropical Heat Spices Chilli Flakes 75G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,False,True,75G,75G,75G,75.0,G
3,32048,6161100911307,Tropical Heat Spices Cardamoms Ground 100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,False,False,100G,100G,100G,100.0,G
4,32114,6161100913080,Tropical Heat Turmeric Ground 45G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,False,False,45G,45G,45G,45.0,G


In [24]:
df['flag_missing_sku'] = df['sku_qty'].isna()
df['flag_zero_or_negative'] = df['sku_qty'] <= 0


In [37]:
# df['flag_missing_sku'] = df['sku_qty'].isna()
# df['flag_zero_or_negative'] = df['sku_qty'] <= 0

print("\n🚩 Missing SKU count:", df['flag_missing_sku'].sum())
print("🚩 Zero/negative SKU count:", df['flag_zero_or_negative'].sum())


🚩 Missing SKU count: 157
🚩 Zero/negative SKU count: 0


In [25]:
df.head()

,id,ean,name,type,food_type,brand,product_origin,supplier,top_category,top_category_id,...,is_fbc,preorder,has_promo,sku,standard_sku,standard_sku_2,sku_qty,sku_unit,flag_missing_sku,flag_zero_or_negative
0,49834,6161100912830,Tropical Heat Spices Pilau Masala Whol100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,100G,100G,100G,100.0,G,False,False
1,32190,6161100913899,Tropical Heat Spices Oregano Leaves 20G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,True,20G,20G,20G,20.0,G,False,False
2,32063,6161100911482,Tropical Heat Spices Chilli Flakes 75G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,True,75G,75G,75G,75.0,G,False,False
3,32048,6161100911307,Tropical Heat Spices Cardamoms Ground 100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,100G,100G,100G,100.0,G,False,False
4,32114,6161100913080,Tropical Heat Turmeric Ground 45G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,False,45G,45G,45G,45.0,G,False,False


In [26]:
df['flag_size_outlier'] = False

df.loc[
    (df['sku_unit'] == 'G') &
    ((df['sku_qty'] < 1) | (df['sku_qty'] > 5000)),
    'flag_size_outlier'
] = True

df.loc[
    (df['sku_unit'] == 'ML') &
    ((df['sku_qty'] < 50) | (df['sku_qty'] > 20000)),
    'flag_size_outlier'
] = True

In [39]:

print("\n🚩 Size outliers:")
print(df.loc[df['flag_size_outlier'],['name', 'sku_qty', 'sku_unit']].head(10))



🚩 Size outliers:
                                                   name  sku_qty sku_unit
746   Chipsy Plus 3 Pure Yellow Vegetable Cooking Fa...  10000.0        G
806                              Tilly Cooking Fat 10Kg  10000.0        G
808               Veebol Cooking Fat Vegetable Oil 10Kg  10000.0        G
2186                         Soko Maize Meal Flour 10Kg  10000.0        G
2291                             Pembe Maize Flour 10Kg  10000.0        G
2442                     Kikkoman Sachet Soy Sauce 30ml     30.0       ML
3363  Britannia Good Day Cashew Cookies 231g x 24 Pi...   5544.0        G
3830                Santa Maria Orange Food Colour 40ml     40.0       ML
3861               Raha Premium Kavagara Maizemeal 10Kg  10000.0        G
3928  Jogoo Maize Meal Fortified With Vitamins And M...  10000.0        G


In [27]:
df.head()

,id,ean,name,type,food_type,brand,product_origin,supplier,top_category,top_category_id,...,preorder,has_promo,sku,standard_sku,standard_sku_2,sku_qty,sku_unit,flag_missing_sku,flag_zero_or_negative,flag_size_outlier
0,49834,6161100912830,Tropical Heat Spices Pilau Masala Whol100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,100G,100G,100G,100.0,G,False,False,False
1,32190,6161100913899,Tropical Heat Spices Oregano Leaves 20G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,True,20G,20G,20G,20.0,G,False,False,False
2,32063,6161100911482,Tropical Heat Spices Chilli Flakes 75G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,True,75G,75G,75G,75.0,G,False,False,False
3,32048,6161100911307,Tropical Heat Spices Cardamoms Ground 100G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,100G,100G,100G,100.0,G,False,False,False
4,32114,6161100913080,Tropical Heat Turmeric Ground 45G,FOOD,DRY,TROPICAL HEAT,Kenya,Carrefour,Food Cupboard,FKEN1700000,...,False,False,45G,45G,45G,45.0,G,False,False,False


In [28]:
def flag_iqr_outliers(group):
    q1 = group.quantile(0.25)
    q3 = group.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ~group.between(lower, upper)

df['flag_iqr_outlier'] = (
    df
    .groupby(['sku_unit'])['sku_qty']
    .transform(flag_iqr_outliers)
)


In [ ]:

print("\n🚩 Unit price outliers:")
print(df.loc[df['flag_unit_price_outlier'], ['name', 'unit_price']].head(10))


In [29]:
df['unit_price'] = df['price'] / df['sku_qty']


In [40]:

print("\n✅ Unit price examples:")
print(df[['name', 'price', 'sku_qty', 'unit_price']].head(10))



✅ Unit price examples:
                                         name  price  sku_qty  unit_price
0  Tropical Heat Spices Pilau Masala Whol100G    285    100.0    2.850000
1     Tropical Heat Spices Oregano Leaves 20G     80     20.0    4.000000
2      Tropical Heat Spices Chilli Flakes 75G    145     75.0    1.933333
3  Tropical Heat Spices Cardamoms Ground 100G    490    100.0    4.900000
4          Tropical Heat  Turmeric Ground 45G     95     45.0    2.111111
5    Tropical Heat Spices Cayenne Pepper 100G    190    100.0    1.900000
6   Naturalli Himalayan Pink Fine Salt 150Gms     99    150.0    0.660000
7               Royco Pilau Masala Spice 100g    309    100.0    3.090000
8      Tropical Heat  Black Pepper Ground 45G    195     45.0    4.333333
9            Tropical Heat  Ginger Ground 45G    115     45.0    2.555556


In [30]:
df['lier'] = False

df.loc[
    (df['unit_price'] <= 0) |
    (df['unit_price'] > df['unit_price'].quantile(0.99)),
    'flag_unit_price_outlier'
] = True

In [32]:
df['flag_any_out'] = (
        df['flag_missing_sku'] |
        df['flag_zero_or_negative'] |
        df['flag_size_outlier'] |
        df['flag_iqr_outlier'] |
        df['flag_unit_price_outlier']
)

In [46]:

print("\n🚩 TOTAL outliers:", df['flag_any_out'].sum())
print("✅ Clean rows:", (~df['flag_any_out']).sum())


🚩 TOTAL outliers: 660
✅ Clean rows: 3340


In [47]:
df_clean = df[~df['flag_any_out']].copy()

In [48]:
reference_price = (
    df_clean
    .groupby(['sku_unit'])['unit_price']
    .median()
)


In [49]:

print("\n✅ Clean data sample:")
print(df_clean[['name', 'standard_sku', 'unit_price']].head(10))


✅ Clean data sample:
                                         name standard_sku  unit_price
0  Tropical Heat Spices Pilau Masala Whol100G         100G    2.850000
1     Tropical Heat Spices Oregano Leaves 20G          20G    4.000000
2      Tropical Heat Spices Chilli Flakes 75G          75G    1.933333
3  Tropical Heat Spices Cardamoms Ground 100G         100G    4.900000
4          Tropical Heat  Turmeric Ground 45G          45G    2.111111
5    Tropical Heat Spices Cayenne Pepper 100G         100G    1.900000
6   Naturalli Himalayan Pink Fine Salt 150Gms         150G    0.660000
7               Royco Pilau Masala Spice 100g         100G    3.090000
8      Tropical Heat  Black Pepper Ground 45G          45G    4.333333
9            Tropical Heat  Ginger Ground 45G          45G    2.555556


In [ ]:
df_clean['unit_price_index'] = (
    df_clean['unit_price'] /
    df_clean['sku_unit'].map(reference_price)
) * 100

In [50]:
reference_price = (
    df_clean
    .groupby('sku_unit')['unit_price']
    .median()
)

print("\n📌 Reference unit prices:")
print(reference_price)

df_clean['unit_price_index'] = (
    df_clean['unit_price'] /
    df_clean['sku_unit'].map(reference_price)
) * 100

print("\n📊 Unit price index sample:")
print(df_clean[['name', 'unit_price', 'unit_price_index']].head(10))


📌 Reference unit prices:
sku_unit
G     1.550000
ML    1.107692
Name: unit_price, dtype: float64

📊 Unit price index sample:
                                         name  unit_price  unit_price_index
0  Tropical Heat Spices Pilau Masala Whol100G    2.850000        183.870968
1     Tropical Heat Spices Oregano Leaves 20G    4.000000        258.064516
2      Tropical Heat Spices Chilli Flakes 75G    1.933333        124.731183
3  Tropical Heat Spices Cardamoms Ground 100G    4.900000        316.129032
4          Tropical Heat  Turmeric Ground 45G    2.111111        136.200717
5    Tropical Heat Spices Cayenne Pepper 100G    1.900000        122.580645
6   Naturalli Himalayan Pink Fine Salt 150Gms    0.660000         42.580645
7               Royco Pilau Masala Spice 100g    3.090000        199.354839
8      Tropical Heat  Black Pepper Ground 45G    4.333333        279.569892
9            Tropical Heat  Ginger Ground 45G    2.555556        164.874552


In [51]:
#saving the cleaned data
df_clean.to_csv("caf_cleaned.csv", index=False)

## further analysis

In [52]:
def parse_category_path(path):
    if pd.isna(path):
        return pd.Series([None, None, None])
    parts = path.split('/')
    return pd.Series([
        parts[0].strip() if len(parts) > 0 else None,
        parts[1].strip() if len(parts) > 1 else None,
        parts[2].strip() if len(parts) > 2 else None
    ])

df_clean[['department', 'category', 'subcategory']] = (
    df_clean['category_path'].apply(parse_category_path)
)

print("✅ Category parsing sample:")
print(df_clean[['category_path', 'department', 'category', 'subcategory']].head(5))

✅ Category parsing sample:
                                       category_path     department  \
0  Food Cupboard/Cooking Ingredients/Herbs, Spice...  Food Cupboard   
1  Food Cupboard/Cooking Ingredients/Herbs, Spice...  Food Cupboard   
2  Food Cupboard/Cooking Ingredients/Herbs, Spice...  Food Cupboard   
3  Food Cupboard/Cooking Ingredients/Herbs, Spice...  Food Cupboard   
4  Food Cupboard/Cooking Ingredients/Herbs, Spice...  Food Cupboard   

              category                subcategory  
0  Cooking Ingredients  Herbs, Spices & Seasoning  
1  Cooking Ingredients  Herbs, Spices & Seasoning  
2  Cooking Ingredients  Herbs, Spices & Seasoning  
3  Cooking Ingredients  Herbs, Spices & Seasoning  
4  Cooking Ingredients  Herbs, Spices & Seasoning  


In [54]:
category_benchmark = (
    df_clean
    .groupby(['category', 'sku_unit'])['unit_price']
    .median()
    .reset_index()
    .rename(columns={'unit_price': 'category_median_unit_price'})
)

print("\n✅ Category median unit prices:")
print(category_benchmark.head(10))


✅ Category median unit prices:
                            category sku_unit  category_median_unit_price
0         Biscuits, Crackers & Cakes        G                    1.071250
1           Breakfast Cereals & Bars        G                    1.441111
2               Chips, Dips & Snacks        G                    1.360390
3          Chocolate & Confectionery        G                    3.500000
4          Chocolate & Confectionery       ML                    8.080000
5  Condiments, Dressings & Marinades        G                    0.707258
6  Condiments, Dressings & Marinades       ML                    1.055882
7                Cooking Ingredients        G                    2.300000
8                Cooking Ingredients       ML                    1.253333
9              Jams, Honey & Spreads        G                    0.831250


In [57]:
df_clean = df_clean.merge(
    category_benchmark,
    on=['category', 'sku_unit'],
    how='left'
)

df_clean['category_price_index'] = (
    df_clean['unit_price'] /
    df_clean['category_median_unit_price']
) * 100

print("\n📊 Category price index sample:")
print(df_clean[['name', 'brand', 'category', 
                'unit_price', 'category_price_index']].head(10))


📊 Category price index sample:
                                         name          brand  \
0  Tropical Heat Spices Pilau Masala Whol100G  TROPICAL HEAT   
1     Tropical Heat Spices Oregano Leaves 20G  TROPICAL HEAT   
2      Tropical Heat Spices Chilli Flakes 75G  TROPICAL HEAT   
3  Tropical Heat Spices Cardamoms Ground 100G  TROPICAL HEAT   
4          Tropical Heat  Turmeric Ground 45G  TROPICAL HEAT   
5    Tropical Heat Spices Cayenne Pepper 100G  TROPICAL HEAT   
6   Naturalli Himalayan Pink Fine Salt 150Gms      NATURALLI   
7               Royco Pilau Masala Spice 100g          ROYCO   
8      Tropical Heat  Black Pepper Ground 45G  TROPICAL HEAT   
9            Tropical Heat  Ginger Ground 45G  TROPICAL HEAT   

              category  unit_price  category_price_index  
0  Cooking Ingredients    2.850000            123.913043  
1  Cooking Ingredients    4.000000            173.913043  
2  Cooking Ingredients    1.933333             84.057971  
3  Cooking Ingredients    4

In [59]:
brand_category_position = (
    df_clean
    .groupby(['category', 'brand'])
    .agg(
        avg_price_index=('category_price_index', 'mean'),
        median_price_index=('category_price_index', 'median'),
        sku_count=('name', 'count')
    )
    .reset_index()
)

print("\n🏷️ Brand price positioning by category:")
print(brand_category_position.head(10))


🏷️ Brand price positioning by category:
                     category        brand  avg_price_index  \
0  Biscuits, Crackers & Cakes       BAKERS       198.366394   
1  Biscuits, Crackers & Cakes      BALOCCO       335.309218   
2  Biscuits, Crackers & Cakes       BERGEN       259.302476   
3  Biscuits, Crackers & Cakes  BETTY CROCK       155.217190   
4  Biscuits, Crackers & Cakes     BRITANIA        90.993954   
5  Biscuits, Crackers & Cakes          C&R       132.022004   
6  Biscuits, Crackers & Cakes      CADBURY       444.033096   
7  Biscuits, Crackers & Cakes       CARR'S       235.239207   
8  Biscuits, Crackers & Cakes    Campiello       303.660817   
9  Biscuits, Crackers & Cakes       DIABLO       497.238429   

   median_price_index  sku_count  
0          198.366394          2  
1          335.309218          1  
2          259.302476          1  
3          117.143707          6  
4           89.105760          4  
5          132.022004          1  
6          444.03309

In [60]:
def price_segment(index):
    if index >= 120:
        return "Premium"
    elif index >= 105:
        return "Upper-Mainstream"
    elif index >= 95:
        return "Mainstream"
    else:
        return "Value"

brand_category_position['price_segment'] = (
    brand_category_position['median_price_index']
    .apply(price_segment)
)

print("\n🏷️ Price segments by brand & category:")
print(brand_category_position[['category', 'brand',
                                'median_price_index',
                                'price_segment']].head(10))



🏷️ Price segments by brand & category:
                     category        brand  median_price_index  \
0  Biscuits, Crackers & Cakes       BAKERS          198.366394   
1  Biscuits, Crackers & Cakes      BALOCCO          335.309218   
2  Biscuits, Crackers & Cakes       BERGEN          259.302476   
3  Biscuits, Crackers & Cakes  BETTY CROCK          117.143707   
4  Biscuits, Crackers & Cakes     BRITANIA           89.105760   
5  Biscuits, Crackers & Cakes          C&R          132.022004   
6  Biscuits, Crackers & Cakes      CADBURY          444.033096   
7  Biscuits, Crackers & Cakes       CARR'S          235.239207   
8  Biscuits, Crackers & Cakes    Campiello          220.642834   
9  Biscuits, Crackers & Cakes       DIABLO          497.238429   

      price_segment  
0           Premium  
1           Premium  
2           Premium  
3  Upper-Mainstream  
4             Value  
5           Premium  
6           Premium  
7           Premium  
8           Premium  
9           P

In [61]:
premium_brands = brand_category_position[
    brand_category_position['price_segment'] == 'Premium'
]

print("\n💎 Premium brands by category:")
print(premium_brands.sort_values(
    ['category', 'median_price_index'],
    ascending=[True, False]
))



💎 Premium brands by category:
                       category           brand  avg_price_index  \
32   Biscuits, Crackers & Cakes  QUALITY STREET       509.218203   
9    Biscuits, Crackers & Cakes          DIABLO       497.238429   
20   Biscuits, Crackers & Cakes         LOACKER       470.805134   
6    Biscuits, Crackers & Cakes         CADBURY       444.033096   
21   Biscuits, Crackers & Cakes           LOTTE       427.021525   
..                          ...             ...              ...   
542          World Specialities           LAY'S       173.724490   
557          World Specialities         SAMYANG       168.421544   
535          World Specialities            ENSO       145.002041   
563          World Specialities        TOP FOOD       143.612245   
548          World Specialities        NONGSHIM       145.975132   

     median_price_index  sku_count price_segment  
32           509.218203          1       Premium  
9            497.238429          1       Premium  

In [ ]:
category_premium_ceiling = (
    brand_category_position
    .groupby('category')['median_price_index']
    .max()
    .reset_index()
    .rename(columns={'median_price_index': 'category_premium_ceiling'})
)

brand_gap = brand_category_position.merge(
    category_premium_ceiling,
    on='category'
)

brand_gap['price_gap_to_premium'] = (
    brand_gap['category_premium_ceiling'] -
    brand_gap['median_price_index']
)

print("\n💸 Brands leaving money on the table:")
print(
    brand_gap
    .sort_values('price_gap_to_premium', ascending=False)
    [['category', 'brand', 'median_price_index',
    'category_premium_ceiling', 'price_gap_to_premium']]
    .head(15)
)


💸 Brands leaving money on the table:
                              category                 brand  \
228  Condiments, Dressings & Marinades              TOP FOOD   
230  Condiments, Dressings & Marinades               Tam Tam   
214  Condiments, Dressings & Marinades                Mama'S   
209  Condiments, Dressings & Marinades           LP - NONAME   
224  Condiments, Dressings & Marinades              SUNFRESH   
191  Condiments, Dressings & Marinades               CLOVERS   
220  Condiments, Dressings & Marinades                PRESTO   
460                Sugar & Home Baking                  SOKO   
441                Sugar & Home Baking                FAMILA   
207  Condiments, Dressings & Marinades                   KOL   
218  Condiments, Dressings & Marinades               PEPTANG   
464                Sugar & Home Baking                 ZESTA   
463                Sugar & Home Baking  WINNIE'S PURE HEALTH   
452                Sugar & Home Baking            MARIANDAZI   
43

In [63]:
summary = (
    brand_gap
    .groupby(['category', 'price_segment'])
    .size()
    .reset_index(name='brand_count')
)

print("\n📊 Brand distribution by price segment:")
print(summary)


📊 Brand distribution by price segment:
                             category     price_segment  brand_count
0          Biscuits, Crackers & Cakes        Mainstream            2
1          Biscuits, Crackers & Cakes           Premium           28
2          Biscuits, Crackers & Cakes  Upper-Mainstream            3
3          Biscuits, Crackers & Cakes             Value           10
4            Breakfast Cereals & Bars        Mainstream            4
5            Breakfast Cereals & Bars           Premium           10
6            Breakfast Cereals & Bars  Upper-Mainstream            5
7            Breakfast Cereals & Bars             Value            7
8                Chips, Dips & Snacks        Mainstream            3
9                Chips, Dips & Snacks           Premium           13
10               Chips, Dips & Snacks  Upper-Mainstream            7
11               Chips, Dips & Snacks             Value           18
12          Chocolate & Confectionery        Mainstream        

## DEFINE PRODUCT‑LEVEL PRICE SEGMENTS (PRINT ✅)

In [65]:
def product_price_segment(index):
    if index >= 120:
        return "Premium"
    elif index >= 105:
        return "Upper-Mainstream"
    elif index >= 95:
        return "Mainstream"
    else:
        return "Value"

df_clean['product_price_segment'] = df_clean['category_price_index'].apply(product_price_segment)

print("✅ Product-level price segments:")
print(df_clean[['name', 'brand', 'category',
                'category_price_index', 'product_price_segment']].head(10))

✅ Product-level price segments:
                                         name          brand  \
0  Tropical Heat Spices Pilau Masala Whol100G  TROPICAL HEAT   
1     Tropical Heat Spices Oregano Leaves 20G  TROPICAL HEAT   
2      Tropical Heat Spices Chilli Flakes 75G  TROPICAL HEAT   
3  Tropical Heat Spices Cardamoms Ground 100G  TROPICAL HEAT   
4          Tropical Heat  Turmeric Ground 45G  TROPICAL HEAT   
5    Tropical Heat Spices Cayenne Pepper 100G  TROPICAL HEAT   
6   Naturalli Himalayan Pink Fine Salt 150Gms      NATURALLI   
7               Royco Pilau Masala Spice 100g          ROYCO   
8      Tropical Heat  Black Pepper Ground 45G  TROPICAL HEAT   
9            Tropical Heat  Ginger Ground 45G  TROPICAL HEAT   

              category  category_price_index product_price_segment  
0  Cooking Ingredients            123.913043               Premium  
1  Cooking Ingredients            173.913043               Premium  
2  Cooking Ingredients             84.057971            

In [66]:
premium_products = df_clean[df_clean['product_price_segment'] == 'Premium']

print("\n💎 Premium products by category:")
print(
    premium_products
    [['category', 'brand', 'name', 'unit_price', 'category_price_index']]
    .sort_values(['category', 'category_price_index'], ascending=[True, False])
    .head(15)
)


💎 Premium products by category:
                        category            brand  \
3236  Biscuits, Crackers & Cakes  Solen:Biscolata   
2127  Biscuits, Crackers & Cakes          CADBURY   
2929  Biscuits, Crackers & Cakes          LOACKER   
1533  Biscuits, Crackers & Cakes   QUALITY STREET   
2740  Biscuits, Crackers & Cakes           DIABLO   
3296  Biscuits, Crackers & Cakes        Campiello   
2317  Biscuits, Crackers & Cakes           NUVITA   
2377  Biscuits, Crackers & Cakes            LOTTE   
2886  Biscuits, Crackers & Cakes          LOACKER   
1503  Biscuits, Crackers & Cakes  Solen:Biscolata   
2655  Biscuits, Crackers & Cakes  Solen:Biscolata   
2334  Biscuits, Crackers & Cakes     GOLDEN BREAK   
2392  Biscuits, Crackers & Cakes     GOLDEN BREAK   
2480  Biscuits, Crackers & Cakes     GOLDEN BREAK   
2552  Biscuits, Crackers & Cakes     GOLDEN BREAK   

                                                   name  unit_price  \
3236                Biscolata Coconut Stix Bisc

In [67]:
#FLAG PRODUCTS LEAVING MONEY ON THE TABLE (🔥 핵심)
# Category premium ceiling at product level
category_product_premium = (
    df_clean
    .groupby('category')['category_price_index']
    .max()
    .reset_index()
    .rename(columns={'category_price_index': 'category_premium_ceiling'})
)

df_clean = df_clean.merge(
    category_product_premium,
    on='category',
    how='left'
)

df_clean['price_gap_to_premium'] = (
    df_clean['category_premium_ceiling'] -
    df_clean['category_price_index']
)

df_clean['flag_money_on_table'] = (
    (df_clean['product_price_segment'] != 'Premium') &
    (df_clean['price_gap_to_premium'] >= 15)
)

print("\n💸 Products leaving money on the table:")
print(
    df_clean[df_clean['flag_money_on_table']]
    [['category', 'brand', 'name',
    'category_price_index', 'category_premium_ceiling',
    'price_gap_to_premium']]
    .sort_values('price_gap_to_premium', ascending=False)
    .head(15)
)


💸 Products leaving money on the table:
                 category    brand  \
2680  Sugar & Home Baking   FAMILA   
3241  Sugar & Home Baking     SOKO   
1744  Sugar & Home Baking   FAMILA   
2841  Sugar & Home Baking  CLOVERS   
1388  Sugar & Home Baking    ZESTA   
1235  Sugar & Home Baking  CLOVERS   
1845  Sugar & Home Baking    ZESTA   
1530  Sugar & Home Baking  CLOVERS   
553   Sugar & Home Baking    ZESTA   
554   Sugar & Home Baking    ZESTA   
552   Sugar & Home Baking  CLOVERS   
563   Sugar & Home Baking  CLOVERS   
1678  Sugar & Home Baking    ZESTA   
1392  Sugar & Home Baking    ZESTA   
549   Sugar & Home Baking    ZESTA   

                                               name  category_price_index  \
2680  Famila The Original Ujimix Sour Porridge 500G             13.307393   
3241        Soko Wimbi Mix Sour Porridge Flour 500g             13.832685   
1744                Famila Pure Wimbi Porridge 500G             15.583658   
2841                        Clovers Corn Fl

In [68]:
#STEP 4 — BRAND × CATEGORY × PRODUCT VIEW (PRINT ✅)
product_segmentation_view = (
    df_clean
    [['category', 'brand', 'name',
    'unit_price', 'category_price_index',
    'product_price_segment', 'flag_money_on_table']]
    .sort_values(['category', 'brand', 'category_price_index'], ascending=[True, True, False])
)

print("\n📊 Product-level segmentation view:")
print(product_segmentation_view.head(20))



📊 Product-level segmentation view:
                        category        brand  \
1587  Biscuits, Crackers & Cakes       BAKERS   
2250  Biscuits, Crackers & Cakes       BAKERS   
2778  Biscuits, Crackers & Cakes      BALOCCO   
2815  Biscuits, Crackers & Cakes       BERGEN   
3303  Biscuits, Crackers & Cakes  BETTY CROCK   
1592  Biscuits, Crackers & Cakes  BETTY CROCK   
1780  Biscuits, Crackers & Cakes  BETTY CROCK   
1948  Biscuits, Crackers & Cakes  BETTY CROCK   
3279  Biscuits, Crackers & Cakes  BETTY CROCK   
2025  Biscuits, Crackers & Cakes  BETTY CROCK   
2631  Biscuits, Crackers & Cakes     BRITANIA   
2671  Biscuits, Crackers & Cakes     BRITANIA   
2516  Biscuits, Crackers & Cakes     BRITANIA   
2343  Biscuits, Crackers & Cakes     BRITANIA   
3179  Biscuits, Crackers & Cakes          C&R   
2127  Biscuits, Crackers & Cakes      CADBURY   
1674  Biscuits, Crackers & Cakes      CADBURY   
2290  Biscuits, Crackers & Cakes       CARR'S   
2400  Biscuits, Crackers & Cakes 

In [69]:
#SUMMARY COUNTS (WHAT LEADERS CARE ABOUT
summary = (
    df_clean
    .groupby(['category', 'product_price_segment'])
    .size()
    .reset_index(name='sku_count')
)

print("\n📈 SKU count by category & price segment:")
print(summary)


📈 SKU count by category & price segment:
                             category product_price_segment  sku_count
0          Biscuits, Crackers & Cakes            Mainstream         19
1          Biscuits, Crackers & Cakes               Premium        122
2          Biscuits, Crackers & Cakes      Upper-Mainstream         28
3          Biscuits, Crackers & Cakes                 Value        147
4            Breakfast Cereals & Bars            Mainstream         20
5            Breakfast Cereals & Bars               Premium         36
6            Breakfast Cereals & Bars      Upper-Mainstream         28
7            Breakfast Cereals & Bars                 Value         62
8                Chips, Dips & Snacks            Mainstream         16
9                Chips, Dips & Snacks               Premium         93
10               Chips, Dips & Snacks      Upper-Mainstream         69
11               Chips, Dips & Snacks                 Value        150
12          Chocolate & Confectione

In [87]:
df_clean.to_csv("caf_cleaned.csv", index=False)

PART A — “SHOULD‑BE PRICE” MODEL (SKU‑LEVEL)
🎯 Objective
For each product:

Estimate a credible higher price it could charge
Anchored in actual category premium behavior
Not hand‑wavy, not “let’s copy the most expensive SKU blindly”


✅ PRINCIPLE (IMPORTANT)
A product’s should‑be price should:

Stay below the category premium ceiling
Respect distance to premium
Avoid over‑stretching value or mainstream SKUs

We do this by closing a controlled % of the gap to premium, not 100%.

In [72]:
#STEP A1 — DEFINE CATEGORY PREMIUM CEILING (RECAP)
category_premium_ceiling = (
    df_clean
    .groupby('category')['category_price_index']
    .max()
    .reset_index()
    .rename(columns={'category_price_index': 'category_premium_ceiling'})
)

print("✅ Category premium ceilings:")
print(category_premium_ceiling.head(10))

✅ Category premium ceilings:
                            category  category_premium_ceiling
0         Biscuits, Crackers & Cakes                583.430572
1           Breakfast Cereals & Bars                791.056284
2               Chips, Dips & Snacks                619.570406
3          Chocolate & Confectionery                388.571429
4  Condiments, Dressings & Marinades               1024.419684
5                Cooking Ingredients                608.695652
6              Jams, Honey & Spreads                740.312319
7         Nuts, Dates & Dried Fruits                765.915845
8               Rice, Pasta & Pulses                713.571429
9                Sugar & Home Baking               1049.708171


In [73]:
#STEP A2 — DEFINE “UPLIFT CAP” RULES (CRITICAL)
def uplift_factor(segment):
    if segment == "Value":
        return 0.4
    elif segment == "Mainstream":
        return 0.6
    elif segment == "Upper-Mainstream":
        return 0.8
    else:  # Premium
        return 0.0


In [75]:
#COMPUTE SHOULD‑BE PRICE INDEX (PRINT ✅)
df_clean['uplift_factor'] = df_clean['product_price_segment'].apply(uplift_factor)

df_clean['should_be_price_index'] = (
    df_clean['category_price_index'] +
    df_clean['uplift_factor'] *
    (df_clean['category_premium_ceiling'] - df_clean['category_price_index'])
)

print("\n📌 Should‑be price index sample:")
print(
    df_clean[['name', 'product_price_segment',
        'category_price_index', 'should_be_price_index']]
    .head(10)
)



📌 Should‑be price index sample:
                                         name product_price_segment  \
0  Tropical Heat Spices Pilau Masala Whol100G               Premium   
1     Tropical Heat Spices Oregano Leaves 20G               Premium   
2      Tropical Heat Spices Chilli Flakes 75G                 Value   
3  Tropical Heat Spices Cardamoms Ground 100G               Premium   
4          Tropical Heat  Turmeric Ground 45G                 Value   
5    Tropical Heat Spices Cayenne Pepper 100G                 Value   
6   Naturalli Himalayan Pink Fine Salt 150Gms                 Value   
7               Royco Pilau Masala Spice 100g               Premium   
8      Tropical Heat  Black Pepper Ground 45G               Premium   
9            Tropical Heat  Ginger Ground 45G      Upper-Mainstream   

   category_price_index  should_be_price_index  
0            123.913043             123.913043  
1            173.913043             173.913043  
2             84.057971             29

In [76]:
#STEP A4 — CONVERT INDEX INTO SHOULD‑BE PRICE (KES) ✅
df_clean['should_be_unit_price'] = (
    df_clean['should_be_price_index'] / 100
) * df_clean['category_price_index'] * df_clean['unit_price'] / df_clean['category_price_index']

df_clean['should_be_price'] = df_clean['should_be_unit_price'] * df_clean['sku_qty']

df_clean['price_uplift_potential'] = df_clean['should_be_price'] - df_clean['price']

print("\n💰 Should‑be price output:")
print(
    df_clean[['name', 'price',
            'should_be_price', 'price_uplift_potential']]
    .head(10)
)


💰 Should‑be price output:
                                         name  price  should_be_price  \
0  Tropical Heat Spices Pilau Masala Whol100G    285       353.152174   
1     Tropical Heat Spices Oregano Leaves 20G     80       139.130435   
2      Tropical Heat Spices Chilli Flakes 75G    145       426.173913   
3  Tropical Heat Spices Cardamoms Ground 100G    490      1043.913043   
4          Tropical Heat  Turmeric Ground 45G     95       283.623188   
5    Tropical Heat Spices Cayenne Pepper 100G    190       556.782609   
6   Naturalli Himalayan Pink Fine Salt 150Gms     99       258.088696   
7               Royco Pilau Masala Spice 100g    309       415.134783   
8      Tropical Heat  Black Pepper Ground 45G    195       367.391304   
9            Tropical Heat  Ginger Ground 45G    115       585.555556   

   price_uplift_potential  
0               68.152174  
1               59.130435  
2              281.173913  
3              553.913043  
4              188.623188  
5

## PART B — PROMO vs BASE‑PRICE DEPENDENCY RISK
This is VERY IMPORTANT for long‑term margin health.

🎯 Objective
Identify products that:

Look cheap only because of frequent promos
Have weak base price positioning
Are at risk if promos are reduced

In [ ]:
#STEP B1 — PROMO DEPENDENCY METRIC (PRINT ✅)
df_clean['promo_discount_pct'] = (
    (df_clean['base_price'] - df_clean['price']) /
    df_clean['base_price']
)

print("\n🏷️ Promo discount sample:")
print(
    df_clean[['product_name', 'base_price', 'price', 'promo_discount_pct']]
    .head(10)
)

In [ ]:
#STEP B2 — BASE vs PROMO PRICE INDEX (KEY INSIGHT)
df_clean['base_unit_price'] = df_clean['base_price'] / df_clean['sku_qty']

df_clean['base_price_index'] = (
    df_clean['base_unit_price'] /
    df_clean['category_median_unit_price']
) * 100

print("\n📊 Base vs promo price index comparison:")
print(
    df_clean[['name', 'base_price_index',
        'category_price_index', 'promo_discount_pct']]
    .head(10)
)

# ✅ Interpretation:

# High base index + heavy discounts → promo‑led premium
# Low base index + heavy discounts → value distortion

In [ ]:
#STEP B3 — PROMO DEPENDENCY RISK FLAG (PRINT ✅)
df_clean['promo_dependency_risk'] = (
    (df_clean['promo_discount_pct'] > 0.25) &
    (df_clean['base_price_index'] < 100)
)

print("\n⚠️ Promo dependency risk SKUs:")
print(
    df_clean[df_clean['promo_dependency_risk']]
    [['category', 'brand', 'name',
    'promo_discount_pct', 'base_price_index']]
    .head(15)
)


In [ ]:
#✅ EXECUTIVE‑READY OUTPUT (ONE TABLE)

final_pricing_view = df_clean[[
    'category', 'brand', 'name',
    'product_price_segment',
    'price', 'should_be_price',
    'price_uplift_potential',
    'promo_discount_pct',
    'promo_dependency_risk'
]]

print("\n📋 Final pricing intelligence view:")
print(final_pricing_view.head(20))

In [79]:
def uplift_factor(segment):
    if segment == "Value":
        return 0.4
    elif segment == "Mainstream":
        return 0.6
    elif segment == "Upper-Mainstream":
        return 0.8
    else:  # Premium
        return 0.0

df_clean['uplift_factor'] = df_clean['product_price_segment'].apply(uplift_factor)

print("✅ Uplift factors by segment:")
print(df_clean[['product_price_segment', 'uplift_factor']].drop_duplicates())

✅ Uplift factors by segment:
   product_price_segment  uplift_factor
0                Premium            0.0
2                  Value            0.4
9       Upper-Mainstream            0.8
15            Mainstream            0.6


In [81]:
df_clean['should_be_price_index'] = (
    df_clean['category_price_index'] +
    df_clean['uplift_factor'] *
    (df_clean['category_premium_ceiling'] - df_clean['category_price_index'])
)

print("\n📊 Should‑be price index examples:")
print(
    df_clean[['name',
              'category_price_index',
              'should_be_price_index',
              'product_price_segment']]
    .head(10)
)


📊 Should‑be price index examples:
                                         name  category_price_index  \
0  Tropical Heat Spices Pilau Masala Whol100G            123.913043   
1     Tropical Heat Spices Oregano Leaves 20G            173.913043   
2      Tropical Heat Spices Chilli Flakes 75G             84.057971   
3  Tropical Heat Spices Cardamoms Ground 100G            213.043478   
4          Tropical Heat  Turmeric Ground 45G             91.787440   
5    Tropical Heat Spices Cayenne Pepper 100G             82.608696   
6   Naturalli Himalayan Pink Fine Salt 150Gms             28.695652   
7               Royco Pilau Masala Spice 100g            134.347826   
8      Tropical Heat  Black Pepper Ground 45G            188.405797   
9            Tropical Heat  Ginger Ground 45G            111.111111   

   should_be_price_index product_price_segment  
0             123.913043               Premium  
1             173.913043               Premium  
2             293.913043            

In [82]:
df_clean['should_be_unit_price'] = (
    df_clean['should_be_price_index'] / df_clean['category_price_index']
) * df_clean['unit_price']

df_clean['should_be_price'] = df_clean['should_be_unit_price'] * df_clean['sku_qty']
df_clean['price_uplift_potential'] = df_clean['should_be_price'] - df_clean['price']

# Avoid negative uplifts (safety)
df_clean.loc[df_clean['price_uplift_potential'] < 0, 'price_uplift_potential'] = 0

print("\n💰 SKU‑level uplift examples:")
print(
    df_clean[['name', 'price',
              'should_be_price', 'price_uplift_potential']]
    .head(10)
)



💰 SKU‑level uplift examples:
                                         name  price  should_be_price  \
0  Tropical Heat Spices Pilau Masala Whol100G    285            285.0   
1     Tropical Heat Spices Oregano Leaves 20G     80             80.0   
2      Tropical Heat Spices Chilli Flakes 75G    145            507.0   
3  Tropical Heat Spices Cardamoms Ground 100G    490            490.0   
4          Tropical Heat  Turmeric Ground 45G     95            309.0   
5    Tropical Heat Spices Cayenne Pepper 100G    190            674.0   
6   Naturalli Himalayan Pink Fine Salt 150Gms     99            899.4   
7               Royco Pilau Masala Spice 100g    309            309.0   
8      Tropical Heat  Black Pepper Ground 45G    195            195.0   
9            Tropical Heat  Ginger Ground 45G    115            527.0   

   price_uplift_potential  
0            0.000000e+00  
1            0.000000e+00  
2            3.620000e+02  
3            5.684342e-14  
4            2.140000e+02 

In [83]:
brand_uplift = (
    df_clean
    .groupby('brand')['price_uplift_potential']
    .sum()
    .reset_index()
    .rename(columns={'price_uplift_potential': 'total_kes_uplift'})
    .sort_values('total_kes_uplift', ascending=False)
)

print("\n💸 TOTAL KES UPSIDE BY BRAND:")
print(brand_uplift)


💸 TOTAL KES UPSIDE BY BRAND:
             brand  total_kes_uplift
165        KAPUTEI     110330.100840
399          ZESTA      95884.659320
94         DORMANS      68989.199611
315    SANTA MARIA      60712.882675
366  TROPICAL HEAT      56044.031429
..             ...               ...
180          KNORR          0.000000
185         KRACKS          0.000000
190        LA MOLE          0.000000
309         SAFCOL          0.000000
200          LOTUS          0.000000

[400 rows x 2 columns]


In [84]:
category_uplift = (
    df_clean
    .groupby('category')['price_uplift_potential']
    .sum()
    .reset_index()
    .rename(columns={'price_uplift_potential': 'total_kes_uplift'})
    .sort_values('total_kes_uplift', ascending=False)
)

print("\n💸 TOTAL KES UPSIDE BY CATEGORY:")
print(category_uplift)


💸 TOTAL KES UPSIDE BY CATEGORY:
                             category  total_kes_uplift
5                 Cooking Ingredients     628334.314203
4   Condiments, Dressings & Marinades     332038.632609
10               Tins, Jars & Packets     316915.956946
9                 Sugar & Home Baking     286036.997675
1            Breakfast Cereals & Bars     222882.600000
6               Jams, Honey & Spreads     134700.305032
7          Nuts, Dates & Dried Fruits     134180.050000
3           Chocolate & Confectionery     119300.969143
2                Chips, Dips & Snacks     105951.628571
0          Biscuits, Crackers & Cakes      95649.950000
8                Rice, Pasta & Pulses      82020.700000
11                 World Specialities      49630.716703


In [85]:
brand_category_uplift = (
    df_clean
    .groupby(['category', 'brand'])
    .agg(
        total_kes_uplift=('price_uplift_potential', 'sum'),
        sku_count=('name', 'count')
    )
    .reset_index()
    .sort_values('total_kes_uplift', ascending=False)
)

print("\n📈 Brand × Category KES Opportunity:")
print(brand_category_uplift.head(20))



📈 Brand × Category KES Opportunity:
                              category                brand  total_kes_uplift  \
437                Sugar & Home Baking              DORMANS      68989.199611   
66            Breakfast Cereals & Bars             WEETABIX      50867.160000   
205  Condiments, Dressings & Marinades              KAPUTEI      42786.646994   
210  Condiments, Dressings & Marinades                LYONS      41505.484658   
234  Condiments, Dressings & Marinades                ZESTA      41196.008434   
267                Cooking Ingredients              KAPUTEI      40462.200000   
484               Tins, Jars & Packets                HOSEN      39853.325000   
433                Sugar & Home Baking              CLOVERS      31548.010249   
322                Cooking Ingredients             TOP FOOD      30775.000000   
68            Breakfast Cereals & Bars            WHITE GLO      30386.600000   
430                Sugar & Home Baking  CAROLINE'S CUPCAKES      28575.2

In [86]:
summary = {
    "Total KES Upside": df_clean['price_uplift_potential'].sum(),
    "Avg KES Upside per SKU": df_clean['price_uplift_potential'].mean(),
    "SKUs with Upside (%)":
        (df_clean['price_uplift_potential'] > 0).mean() * 100
}

print("\n📊 EXECUTIVE SUMMARY:")
for k, v in summary.items():
    print(f"{k}: {round(v,2)}")


📊 EXECUTIVE SUMMARY:
Total KES Upside: 2507642.82
Avg KES Upside per SKU: 750.79
SKUs with Upside (%): 64.82
